# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/manahilrubabsatti/flyrank-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [10]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF_TOKEN loaded:", HF_TOKEN is not None)

HF_TOKEN loaded: True


In [11]:
# ML-04 — Warehouse setup

from google.colab import userdata
import duckdb

# Get the Hugging Face token from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

# Connect to DuckDB
con = duckdb.connect()

# Authenticate with Hugging Face
con.execute(
    "CREATE SECRET (TYPE huggingface, TOKEN ?)",
    [HF_TOKEN]
)

# FlyRank warehouse
rel = "hf://datasets/FlyRank/internship-warehouse"

print("Warehouse connection ready.")

Warehouse connection ready.


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

For my Refresh / Content Opportunity Scoring lane, one row represents one content item for one client on one report date. I will use March 2026 as the development month. I will aggregate the daily observations to the content-item level when creating my features and ranking pages for a possible content refresh.

In [12]:
# Verify the actual warehouse grain using March 2026

query = """
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_data_available,
    ga4_data_available,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_pageviews,
    ga4_sessions
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
LIMIT 10
"""

sample = con.sql(query).df()

print("Sample March 2026 rows:")
display(sample)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Sample March 2026 rows:


,report_date,client_hash_id,content_hash_id,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,<NA>,20,0,3.350000,<NA>,<NA>
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,<NA>,1,0,0.000000,<NA>,<NA>
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,<NA>,125,1,4.928000,<NA>,<NA>
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,<NA>,7,0,4.000000,<NA>,<NA>
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,<NA>,11,0,2.272727,<NA>,<NA>
5,2026-03-01,client_73cda7b4e4f265ea,content_36c36abc7650d7af,True,<NA>,239,1,7.347280,<NA>,<NA>
6,2026-03-01,client_73cda7b4e4f265ea,content_a7da352b73b02668,True,<NA>,191,0,7.832461,<NA>,<NA>
7,2026-03-01,client_73cda7b4e4f265ea,content_05434271b257bb68,True,<NA>,55,0,3.272727,<NA>,<NA>
8,2026-03-01,client_73cda7b4e4f265ea,content_d056587ff7faca0c,True,<NA>,77,0,5.636364,<NA>,<NA>
9,2026-03-01,client_73cda7b4e4f265ea,content_bfd1e41c2af250c8,True,<NA>,2,0,4.500000,<NA>,<NA>


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features

The main features I plan to use are:
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_pageviews
- ga4_sessions

These describe observed search and analytics performance that can be available when making a content-review decision.

### Label / outcome

The eventual outcome will be a future performance or decline signal defined using a later time window. The label should only use information from after the decision point so that the features do not contain future information.

### Context

I will keep report_date, client_hash_id, and content_hash_id as context fields for grouping, filtering, and identifying the page-level observation.

### Excluded

I will exclude client-identifying information, raw private queries, and any information that would reveal a client or domain. I will also exclude future outcome information from the features to avoid leakage.

In [13]:
# Check that the planned feature columns exist

features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions"
]

print("Planned features:")
for feature in features:
    print("-", feature)

Planned features:
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_pageviews
- ga4_sessions


In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Query 1 — Grain

I expect each warehouse row to represent one client, one content item, and one report date. I will verify this by checking whether the combination of these three fields is unique in the selected month.

In [15]:
# Query 1: verify whether client + content + date identifies a row

query_grain = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT
        client_hash_id || '|' ||
        content_hash_id || '|' ||
        CAST(report_date AS VARCHAR)
    ) AS unique_client_content_date_rows
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
"""

grain_check = con.sql(query_grain).df()

display(grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_client_content_date_rows
0,9841378,9841378


### Query 2 — March 2026 coverage

I will check how many rows are available in March 2026 and confirm the earliest and latest report dates in this development window.

In [16]:
# Query 2: row count and date span

query_dates = """
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
"""

date_check = con.sql(query_dates).df()

display(date_check)

,row_count,first_date,last_date
0,9841378,2026-03-01,2026-03-31


### Query 3 — Data availability

For the content-refresh lane, I want to know how many March observations have GSC data available. I use `IS TRUE` so that only rows explicitly marked as available are counted.

In [17]:
# Query 3: GSC availability using IS TRUE

query_availability = """
SELECT
    COUNT(*) AS march_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS rows_with_gsc_data
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
"""

availability_check = con.sql(query_availability).df()

display(availability_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,march_rows,rows_with_gsc_data
0,9841378,3611061


In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Five-feature frame

For the first version of the lane, I will use five observed performance features. These features summarize search visibility, search clicks, search position, and analytics engagement.

- **gsc_impressions** — available when the GSC data for the observation period is available.
- **gsc_clicks** — available when the GSC data for the observation period is available.
- **gsc_avg_position** — available when GSC position data is available.
- **ga4_pageviews** — available when GA4 data is available.
- **ga4_sessions** — available when GA4 data is available.

These features are intended to describe the page at the decision moment rather than include information from the future outcome window.

In [19]:
# Build a small March 2026 feature frame

feature_query = """
SELECT
    client_hash_id,
    content_hash_id,

    SUM(gsc_impressions) AS gsc_impressions,
    SUM(gsc_clicks) AS gsc_clicks,
    AVG(gsc_avg_position) AS gsc_avg_position,
    SUM(ga4_pageviews) AS ga4_pageviews,
    SUM(ga4_sessions) AS ga4_sessions

FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)

WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'

GROUP BY
    client_hash_id,
    content_hash_id

LIMIT 20
"""

feature_frame = con.sql(feature_query).df()

print("March 2026 feature frame:")
display(feature_frame)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March 2026 feature frame:


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6523.0,7.0,7.209549,1.0,1.0
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,453.0,0.0,2.987198,0.0,0.0
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5630.0,6.0,6.724039,6.0,3.0
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,4944.0,13.0,7.244844,2.0,2.0
4,client_73cda7b4e4f265ea,content_f39be42b42a4e8f6,42.0,0.0,14.432540,8.0,7.0
5,client_73cda7b4e4f265ea,content_1855a661b4d36130,429.0,1.0,4.209227,2.0,2.0
6,client_73cda7b4e4f265ea,content_5d412fba6e1a2582,223.0,1.0,9.445635,2.0,2.0
7,client_73cda7b4e4f265ea,content_1f380a642aed423b,96.0,1.0,6.014516,10.0,8.0
8,client_73cda7b4e4f265ea,content_22c063002b7c1caf,314.0,1.0,9.155335,0.0,0.0
9,client_73cda7b4e4f265ea,content_aafb2ab7e5fc80d0,7709.0,20.0,5.258331,12.0,12.0


### Deliberate leakage experiment

To demonstrate leakage, I will intentionally create a feature that is directly derived from the outcome. This should make the relationship look unrealistically strong. This is not a valid feature because the information would not be available at the time the decision is made.

After demonstrating the problem, I will remove the leaked feature and keep only information that would genuinely be available at the decision moment.

In [20]:
# Deliberate leakage demonstration
# Here the "leaky feature" is literally the outcome we are trying to describe.

leak_demo = feature_frame.copy()

leak_demo["outcome"] = (
    leak_demo["gsc_impressions"] > leak_demo["gsc_impressions"].median()
).astype(int)

# This feature directly copies the outcome.
leak_demo["LEAKED_FEATURE"] = leak_demo["outcome"]

print("Leakage check:")
print(
    "Rows where leaked feature equals outcome:",
    (leak_demo["LEAKED_FEATURE"] == leak_demo["outcome"]).sum(),
    "out of",
    len(leak_demo)
)

Leakage check:
Rows where leaked feature equals outcome: 20 out of 20


### Leakage removed

The leaked feature is removed from the final feature set. It would not be available at the decision moment and would give an unrealistically strong result. The final model should only use information that is available before the future outcome is observed.

In [21]:
# Remove the deliberately leaked feature

honest_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions"
]

final_feature_frame = leak_demo[
    ["client_hash_id", "content_hash_id"] + honest_features
].copy()

print("Final honest feature columns:")
print(final_feature_frame.columns.tolist())

display(final_feature_frame.head())

Final honest feature columns:
['client_hash_id', 'content_hash_id', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions']


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6523.0,7.0,7.209549,1.0,1.0
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,453.0,0.0,2.987198,0.0,0.0
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5630.0,6.0,6.724039,6.0,3.0
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,4944.0,13.0,7.244844,2.0,2.0
4,client_73cda7b4e4f265ea,content_f39be42b42a4e8f6,42.0,0.0,14.432540,8.0,7.0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This data is useful for observed and directional decision support, but it has important limits. The history is an unbalanced panel, so not every client or content item has the same amount of historical data. GSC and GA4 availability can also differ between observations. Search and analytics metrics describe what was observed, but they do not prove that a content change caused a performance change. The final month should also be treated carefully because it can represent the future outcome window for models developed on earlier months.

Therefore, I will use the data to identify pages that may deserve review, not to claim causal effects or predict exactly what Google will do.

In [22]:
# Check how many March rows have GSC availability

gsc_summary = con.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE client_has_gsc IS TRUE) AS rows_with_gsc,
    COUNT(*) FILTER (WHERE client_has_gsc IS NOT TRUE) AS rows_without_gsc
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
""").df()

display(gsc_summary)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,rows_with_gsc,rows_without_gsc
0,9841378,9841378,0


In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.